In [1]:
import numpy as np
import pandas as pd
import ast
import re

In [2]:
df_men_review = pd.read_csv('./data/xexymix_raw_review.csv')

## 1. 젝시믹스 분석 기간인 2024년 이후로

In [3]:
# #2024년 이후
df_men_review['year'] = pd.to_datetime(df_men_review['created_at'],format = 'mixed', utc=True).dt.year
print(df_men_review['year'].value_counts().sort_index())

df_men_review = df_men_review[df_men_review['year']>=2024]
#원본 : 343901 , 2024년 이후 : 232,476 
print(df_men_review['year'].value_counts().sort_index())

year
2015         6
2016        66
2017       269
2018      2244
2019      3296
2020     14153
2021    150272
2022    342388
2023    551254
2024    546147
2025    633199
2026    120808
Name: count, dtype: int64
year
2024    546147
2025    633199
2026    120808
Name: count, dtype: int64


## 2. collect_date = 2026.04.20

In [4]:
df_men_review["collect_date"] = pd.to_datetime("2026-04-20")
df_men_review["collect_date"] = df_men_review["collect_date"].dt.date

## 3. created_at > post_date

In [5]:
# 날짜 형식으로 변환
df_men_review["created_at"] = pd.to_datetime(
    df_men_review["created_at"], 
    format="ISO8601"
)

# 시간 정보를 제외한 날짜만 추출
df_men_review["created_at"] = df_men_review["created_at"].dt.date

# 컬럼명 변경
df_men_review = df_men_review.rename(columns={"created_at": "post_date"})

In [6]:
df_men_review['post_date']

0          2026-03-09
1          2026-01-26
2          2025-12-14
3          2025-07-22
4          2025-10-10
              ...    
2364023    2024-04-16
2364027    2024-05-30
2364057    2026-03-21
2364065    2025-05-03
2364074    2024-09-29
Name: post_date, Length: 1300154, dtype: object

In [7]:
# # 날짜 형식으로 변환 (이 작업이 반드시 먼저 필요합니다)
# df_men_review["created_at"] = pd.to_datetime(df_men_review["created_at"])

# # 시간 정보를 제외한 날짜만 추출
# df_men_review["created_at"] = df_men_review["created_at"].dt.date

# # 컬럼명 변경
# df_men_review = df_men_review.rename(columns={"created_at": "post_date"})

## 4. platform  = 자사몰 생성

In [8]:
df_men_review['platform'] = '자사몰'

## 5. filtered_message > content

In [9]:
df_men_review = df_men_review.rename(columns={"filtered_message": "content"})

## 6. product_url > source_url

In [10]:
df_men_review = df_men_review.rename(columns={"product_url": "source_url"})

## 7. score > rating

In [11]:
df_men_review = df_men_review.rename(columns={"score": "rating"})

## 8.likes_count(plus_likes_count) > helpful_count

In [12]:
df_men_review = df_men_review.rename(columns={"likes_count": "helpful_count"})

## 9.product_options > purchased_option

In [13]:
df_men_review = df_men_review.rename(columns={"product_options": "purchased_option"})

## 10.product_options 기반 has_image 칼럼 생성

In [14]:
df_men_review["has_image"] = df_men_review["images"].apply(bool).astype(int)

In [15]:
df_men_review.info()

<class 'pandas.DataFrame'>
Index: 1300154 entries, 0 to 2364074
Data columns (total 46 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   id                              1300154 non-null  int64  
 1   rating                          1300154 non-null  int64  
 2   store_name                      0 non-null        float64
 3   brand_user_id                   1216384 non-null  float64
 4   user_display_name               1300154 non-null  str    
 5   media_count                     1300154 non-null  int64  
 6   product_code                    1300154 non-null  int64  
 7   product_image_source_url        1300154 non-null  str    
 8   product_image_url               1300154 non-null  str    
 9   product_name                    1300154 non-null  str    
 10  product_meta_reviews_count      1300154 non-null  int64  
 11  product_meta_score              1300154 non-null  float64
 12  social_media_typ

## 스키마 칼럼만

In [16]:
df_men_review["search_keyword"] = None

In [17]:
df_men_review.columns

Index(['id', 'rating', 'store_name', 'brand_user_id', 'user_display_name',
       'media_count', 'product_code', 'product_image_source_url',
       'product_image_url', 'product_name', 'product_meta_reviews_count',
       'product_meta_score', 'social_media_type', 'source_url', 'post_date',
       'editable', 'deletable', 'tags', 'helpful_count', 'plus_likes_count',
       'reported', 'my_review', 'author_grade', 'like_score', 'comments_count',
       'review_vendor', 'external_platform_type', 'review_source',
       'review_source_has_video', 'content', 'content_type',
       'review_source_url', 'verified_buyer_badge_image_url', 'ai_summary',
       'brand_user_grade', 'videos', 'images', 'purchased_option',
       'customer_properties', 'brand_author_type', 'category', 'brand', 'year',
       'collect_date', 'platform', 'has_image', 'search_keyword'],
      dtype='str')

- 키, 몸무게 추출

In [18]:
def parse_str_to_list(x):
    if not x or x == '[]':
        return []
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

df_men_review['customer_properties_parsed'] = df_men_review['customer_properties'].apply(parse_str_to_list)

In [19]:
df_men_review['customer_properties_values'] = df_men_review['customer_properties_parsed'].apply(
    lambda lst: [d.get('value') for d in lst] if lst else []
)

In [20]:
def extract_number(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r'\d+\.?\d*', str(value))
    return float(match.group()) if match else np.nan

df_men_review['user_height'] = df_men_review['customer_properties_values'].apply(lambda x: x[0] if len(x) > 0 else np.nan)
df_men_review['user_weight'] = df_men_review['customer_properties_values'].apply(lambda x: extract_number(x[1]) if len(x) > 1 else np.nan)
df_men_review['user_size'] = df_men_review['customer_properties_values'].apply(lambda x: x[2] if len(x) > 2 else np.nan)

df_men_review['user_height'] = df_men_review['user_height'].str.replace('cm', '', regex=False)

In [22]:
final_men_review_df = df_men_review[[
    "id",
    "collect_date",
    "post_date",
    "platform",
    "brand",
    "product_code",
    "product_name",
    "search_keyword",
    "content",
    "source_url",
    "rating",
    "helpful_count",
    "purchased_option",
    "has_image",
    "user_height",
    "user_weight",
    "user_size"
]]

In [23]:
final_men_review_df.to_csv(
    "xexymix_raw_review_final.csv",
    index=False,
    encoding="utf-8-sig"
)

In [24]:
final_men_review_df.info()

<class 'pandas.DataFrame'>
Index: 1300154 entries, 0 to 2364074
Data columns (total 17 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   id                1300154 non-null  int64  
 1   collect_date      1300154 non-null  object 
 2   post_date         1300154 non-null  object 
 3   platform          1300154 non-null  str    
 4   brand             1300154 non-null  str    
 5   product_code      1300154 non-null  int64  
 6   product_name      1300154 non-null  str    
 7   search_keyword    0 non-null        object 
 8   content           1300154 non-null  str    
 9   source_url        1300154 non-null  str    
 10  rating            1300154 non-null  int64  
 11  helpful_count     1300154 non-null  int64  
 12  purchased_option  1300154 non-null  str    
 13  has_image         1300154 non-null  int64  
 14  user_height       1163780 non-null  str    
 15  user_weight       1126632 non-null  float64
 16  user_size       

In [ ]:
df_men_review['product_image_url'].isnull().sum()

np.int64(0)